In [1]:
import os
os.chdir("../../")
print("Current working directory:", os.getcwd())
import csv
import nest_asyncio
import pandas as pd

nest_asyncio.apply()
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core import Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, AsyncQdrantClient
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.llms.openai_like import OpenAILike
from IPython.display import display, Markdown
import json
from typing import List

# By pass SSL certification
import httpx

# Apply the monkey patch
from chainlit_app.patches import patch

patch.apply_patch()

Current working directory: /data/ai-chatbot


/data/ai-chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2025-01-22 11:22:53,642 - INFO - Patched TextEmbeddingsInference._call_api with custom synchronous API handling
2025-01-22 11:22:53,643 - INFO - Patched TextEmbeddingsInference._acall_api with custom asynchronous API handling


In [2]:
# Trace llamaindex
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register(
    project_name="test-indexing",
    endpoint="http://localhost:3000/v1/traces",
)

LlamaIndexInstrumentor().instrument(
    skip_dep_check=True, tracer_provider=tracer_provider
)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: test-indexing
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:3000/v1/traces
|  Transport: HTTP
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
def create_faq_docs(csv_path: str, documents: List[Document]) -> int:
    initial_count = len(documents)  # Track initial number of documents
    with open(csv_path, mode="r", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)

        # Normalize column names: strip spaces and remove spaces in the middle
        reader.fieldnames = ["".join(field.split()) for field in reader.fieldnames]
        
        print("FAQs Column Name:", reader.fieldnames)

        for i, row in enumerate(reader):
            if row["AnswerTH(FriendlyEmoji)"].strip():
                # Create metadata for the documents
                base_metadata = {
                    # "source": "FAQ Dataset",
                    # "row_index": i,
                    "language": "Thai",  # Default language, updated per document
                }
                # Create unique doc_id for Thai (TH) document
                doc_id_th = row["doc_id_th"].strip()
                json_data_th = {
                    "Question Example": row["QuestionTH"].strip(),
                    "Answer Example": row["AnswerTH(FriendlyEmoji)"].strip(),
                    "Question Type": row["Type"].strip(),
                }
                metadata_th = base_metadata.copy()
                metadata_th.update(
                    {
                        "language": "Thai", 
                        # "online_offline": row["Online/Offline"].strip()
                    }
                )
                text_content_th = json.dumps(json_data_th, ensure_ascii=False)
                document_th = Document(
                    text=text_content_th, 
                    metadata=metadata_th, 
                    doc_id=doc_id_th
                )
                documents.append(document_th)

            if row["AnswerEN(FriendlyEmoji)"].strip():
                # Create unique doc_id for English (EN) document
                doc_id_en = row["doc_id_en"].strip()
                json_data_en = {
                    "Question Example": row["QuestionEN"].strip(),
                    "Answer Example": row["AnswerEN(FriendlyEmoji)"].strip(),
                    "Question Type": row["Type"].strip(),
                }
                metadata_en = base_metadata.copy()
                metadata_en.update(
                    {   
                        "language": "English",
                        # "online_offline": row["Online/Offline"].strip()
                    }
                )
                text_content_en = json.dumps(json_data_en, ensure_ascii=False)
                document_en = Document(
                    text=text_content_en,
                    metadata=metadata_en,
                    doc_id=doc_id_en
                )
                documents.append(document_en)

    # Return the number of documents appended
    return len(documents) - initial_count

def convert_to_time_format(raw_time: str) -> str:
    """Convert raw time (e.g., '700') to time format (e.g., '07:00')."""
    return f"{raw_time[:-2].zfill(2)}:{raw_time[-2:]}"

def create_store_docs_with_time_format(file_path: str, documents: List[Document]) -> int:
    """
    Create store documents with time format support from a file (CSV or Excel).
    
    Args:
        file_path (str): Path to the input file (CSV or Excel).
        documents (List[Document]): List to which the documents will be appended.
    
    Returns:
        int: Number of documents appended.
    """
    initial_count = len(documents)  # Track initial number of documents

    # Determine file type and read the file
    if file_path.endswith('.csv'):
        data = pd.read_csv(file_path)
    elif file_path.endswith(('.xls', '.xlsx')):
        data = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Only CSV and Excel files are allowed.")

    # print("Storeline Column Name Before:", data.columns)
    # Normalize column names: strip spaces and remove spaces in the middle
    data.columns = ["".join(col.split()) for col in data.columns]
    
    print("Storeline Column Name After:", data.columns)

    for _, row in data.iterrows():
        # Clean each field in the row by trimming spaces and removing quotes
        cleaned_row = {key: str(value).strip().strip('"') for key, value in row.items()}

        # Create doc_id for each document
        doc_id = cleaned_row["doc_id"]

        # Create JSON structure
        json_data = {
            # "Store ID": cleaned_row["StoreID"],
            "StoreName Thai": cleaned_row["Name"],
            "StoreName English": cleaned_row["EnglishName"],
            "Latitude": float(cleaned_row["Latitude"]),
            "Longitude": float(cleaned_row["Longitude"]),
            "Location": cleaned_row["GoogleMapURL"],
            # "Online Status": cleaned_row["OnlineStatus"],
            "Store Type": cleaned_row["StoreType"],
            # "Street": cleaned_row["Street"],
            # "City": cleaned_row["City"],
            # "State": cleaned_row["State"],
            # "Postcode": cleaned_row["Postcode"],
            # "Region": cleaned_row["region"],
            # "Delivery Type": cleaned_row["DeliveryType"],
            "Format": cleaned_row["OfflineFormat"],
            "Open Hour (เวลาเปิด-ปิด)": cleaned_row["OfflineOpenHour"],
            "Online Delivery Service Hour": cleaned_row["OnlineOpenHour"],
            "Address (ที่อยู่)": cleaned_row["OfflineAddress+Phone"],
            "Store Email": cleaned_row["StoreEmail"],
            # "Contact Number": cleaned_row["RevisedContactNumber"]
        }

        # Metadata for the document
        metadata = {
            # "source": "Store Dataset",
            # "store_id": cleaned_row["StoreID"],
            # "region": cleaned_row["region"],
            # "online_status": cleaned_row["OnlineStatus"]
        }

        # Create Document object
        text_content = json.dumps(json_data, ensure_ascii=False)
        document = Document(text=text_content, metadata=metadata, doc_id=doc_id)
        documents.append(document)

    # Return the number of documents appended
    return len(documents) - initial_count

In [4]:
# Create an empty documents list
documents = []

# Append FAQ documents and report the count
faq_count = create_faq_docs("documents/FAQ_Update_210225_update.csv", documents)
print(f"{faq_count} FAQ documents have been appended.")

# Append Store documents and report the count
store_count = create_store_docs_with_time_format("documents/Store_Info_140125.csv", documents)
print(f"{store_count} Store documents have been appended.")

# Print the total number of documents
print(f"Total documents: {len(documents)}")

FAQs Column Name: ['index', 'doc_id_th', 'doc_id_en', 'QuestionTH', 'QuestionEN', 'AnswerTH', 'AnswerTH(Friendly)', 'AnswerEN', 'AnswerTH(FriendlyEmoji)', 'AnswerEN(Friendly)', 'AnswerEN(FriendlyEmoji)', 'Type', 'Online/Offline']
313 FAQ documents have been appended.
Storeline Column Name After: Index(['Storeid', 'StoreID', 'doc_id', 'Name', 'EnglishName', 'Latitude',
       'Longitude', 'GoogleMapURL', 'OnlineStatus', 'StoreType', 'Street',
       'City', 'State', 'Postcode', 'region', 'DeliveryType', 'OnlineOpenHour',
       'OnlineOpenHour.1', 'OfflineFormat', 'OfflineOpenHour',
       'OfflineAddress', 'OfflineAddress+Phone', 'StoreEmail', 'ContactNumber',
       'ContactNumber.1', 'Unnamed:25', 'Unnamed:26', 'Unnamed:27',
       'Valid/Invalid', 'Unnamed:29', 'RevisedContactNumber'],
      dtype='object')
2543 Store documents have been appended.
Total documents: 2856


In [5]:
def print_documents_by_indices(documents: List[Document], indices: List[int]) -> None:
    """
    Print documents at specific indices.

    Parameters:
        documents (List[Document]): The list of documents.
        indices (List[int]): A list of indices of the documents to print.
    """
    for index in indices:
        if 0 <= index < len(documents):  # Check if index is within bounds
            print(f"Document {index}:")
            print(f"  doc_id: {documents[index].doc_id}")
            print(f"  text: {documents[index].text}")
            print(f"  metadata: {documents[index].metadata}")
            print()  # Add a blank line for readability
        else:
            print(f"Index {index} is out of bounds. Please specify a valid index.")
            
# Specify the list of indices to print
indices_to_print = [0, 1]

# Print the documents at the specified indices
print_documents_by_indices(documents, indices_to_print)

# Specify the list of indices to print
indices_to_print = [400, 401]

# Print the documents at the specified indices
print_documents_by_indices(documents, indices_to_print)

Document 0:
  doc_id: doc_id_th_1
  text: {"Question Example": "สามารถชำระ Bill Payment ด้วยบัตรเครดิตได้หรือไม่", "Answer Example": "สำหรับการชำระ Bill Payment คุณลูกค้าสามารถชำระได้ด้วยเงินสดเท่านั้นค่ะ 😊", "Question Type": "Bill Payment"}
  metadata: {'language': 'Thai'}

Document 1:
  doc_id: doc_id_en_1
  text: {"Question Example": "Can I pay the bill with a credit card?", "Answer Example": "For Bill Payment, customers can pay with cash only 😊.", "Question Type": "Bill Payment"}
  metadata: {'language': 'English'}

Document 400:
  doc_id: store_doc_1211
  text: {"StoreName Thai": "โลตัส โกเฟรช บ้านสวนธน", "StoreName English": "Lotus Go Fresh Ban Soanthon", "Latitude": 13.65126, "Longitude": 100.48733, "Location": "https://www.google.com/maps/place/13.65126,100.48733", "Store Type": "go_fresh", "Format": "Mini Supermarket", "Open Hour (เวลาเปิด-ปิด)": "24 Hours", "Online Delivery Service Hour": "7:00-22:00", "Address (ที่อยู่)": "เลขที่ 365/1644,365/1646 หมู่ 2 แขวงบางมด เขตทุ่งครุ

In [6]:
# Specify the list of indices to print
indices_to_print = [1000]

# Print the documents at the specified indices
print_documents_by_indices(documents, indices_to_print)

Document 1000:
  doc_id: store_doc_2113
  text: {"StoreName Thai": "โลตัส โกเฟรช ตลาดวัดศรีบุญยืน ลำพูน", "StoreName English": "Lotus Go Fresh Talad Wat Sriboonyuen", "Latitude": 18.599306, "Longitude": 99.027154, "Location": "https://www.google.com/maps/place/18.599306,99.027154", "Store Type": "go_fresh", "Format": "Mini Supermarket", "Open Hour (เวลาเปิด-ปิด)": "06:00-23:00 (17Hrs)", "Online Delivery Service Hour": "7:00-20:00", "Address (ที่อยู่)": "เลขที่ 167/2 หมู่ที่ 3 ตำบลเหมืองง่า อำเภอเมืองลำพูน จังหวัดลำพูน 51000, โทรศัพท์: 062-605-8247", "Store Email": "exp2113-sm@lotuss.com"}
  metadata: {}



In [7]:
# Create an httpx client with SSL verification disabled
http_client = httpx.Client(verify=False)
# Create an httpx AsyncClient with SSL verification using certifi
async_http_client = httpx.AsyncClient(verify=False)

# Set up LLM model
Settings.llm = OpenAILike(
    model="default",
    api_base="https://api.cpxis.global.lotuss.org/llm//v1",
    api_key="automation.lotuss.Zb71t4pjNR3rty3uI8os9jwxaJmU8h",
    is_chat_model=True,
    is_function_calling_model=False,
    temperature=0.2,
    http_client=http_client,
    async_http_client=async_http_client,
    # max_tokens=4096,
)

In [8]:
# Initialize the embedding settings
embed_model = TextEmbeddingsInference(
    model_name="BAAI/bge-m3",
    base_url=f"https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3",
    auth_token=f"Bearer automation.lotuss.Zb71t4pjNR3rty3uI8os9jwxaJmU8h",
    timeout=60,
    embed_batch_size=10,
)

Settings.embed_model = embed_model

In [9]:
from llama_index.core.vector_stores import VectorStoreQueryResult


def relative_score_fusion(
    dense_result: VectorStoreQueryResult,
    sparse_result: VectorStoreQueryResult,
    alpha: float = 0.5,  # passed in from the query engine
    top_k: int = 2,  # passed in from the query engine i.e. similarity_top_k
) -> VectorStoreQueryResult:
    """
    Fuse dense and sparse results using relative score fusion.
    """
    print("Using Qdrant Hybrid Search: alpha:", alpha)
    # sanity check
    assert dense_result.nodes is not None
    assert dense_result.similarities is not None
    assert sparse_result.nodes is not None
    assert sparse_result.similarities is not None

    # deconstruct results
    sparse_result_tuples = list(
        zip(sparse_result.similarities, sparse_result.nodes)
    )
    sparse_result_tuples.sort(key=lambda x: x[0], reverse=True)

    dense_result_tuples = list(
        zip(dense_result.similarities, dense_result.nodes)
    )
    dense_result_tuples.sort(key=lambda x: x[0], reverse=True)

    # track nodes in both results
    all_nodes_dict = {x.node_id: x for x in dense_result.nodes}
    for node in sparse_result.nodes:
        if node.node_id not in all_nodes_dict:
            all_nodes_dict[node.node_id] = node

    # normalize sparse similarities from 0 to 1
    sparse_similarities = [x[0] for x in sparse_result_tuples]
    max_sparse_sim = max(sparse_similarities)
    min_sparse_sim = min(sparse_similarities)
    sparse_similarities = [
        (x - min_sparse_sim) / (max_sparse_sim - min_sparse_sim)
        for x in sparse_similarities
    ]
    sparse_per_node = {
        sparse_result_tuples[i][1].node_id: x
        for i, x in enumerate(sparse_similarities)
    }

    # normalize dense similarities from 0 to 1
    dense_similarities = [x[0] for x in dense_result_tuples]
    max_dense_sim = max(dense_similarities)
    min_dense_sim = min(dense_similarities)
    dense_similarities = [
        (x - min_dense_sim) / (max_dense_sim - min_dense_sim)
        for x in dense_similarities
    ]
    dense_per_node = {
        dense_result_tuples[i][1].node_id: x
        for i, x in enumerate(dense_similarities)
    }

    # fuse the scores
    fused_similarities = []
    for node_id in all_nodes_dict:
        sparse_sim = sparse_per_node.get(node_id, 0)
        dense_sim = dense_per_node.get(node_id, 0)
        fused_sim = alpha * (sparse_sim + dense_sim)
        fused_similarities.append((fused_sim, all_nodes_dict[node_id]))

    fused_similarities.sort(key=lambda x: x[0], reverse=True)
    fused_similarities = fused_similarities[:top_k]

    # create final response object
    return VectorStoreQueryResult(
        nodes=[x[1] for x in fused_similarities],
        similarities=[x[0] for x in fused_similarities],
        ids=[x[1].node_id for x in fused_similarities],
    )

In [10]:
api_key = "QdrantVAsfhF8nGPtyleJKVkt2TBI2bqQ4bSjgnajNtOLwE2Y9YWxnZFrItRBE53"
# creates a persistant index to disk
client = QdrantClient(url="http://localhost:6334", api_key=api_key,  prefer_grpc=True)
aclient = AsyncQdrantClient(url="http://localhost:6334", api_key=api_key, prefer_grpc=True)

# # delete collection if it exists
if client.collection_exists("vector_data"):
    client.delete_collection("vector_data")

# create our vector store with hybrid indexing enabled
# batch_size controls how many nodes are encoded with sparse vectors at once
vector_store = QdrantVectorStore(
    "vector_data",
    client=client,
    aclient=aclient,
    enable_hybrid=True,
    batch_size=20,
    prefer_grpc=True,
    hybrid_fusion_fn=relative_score_fusion,
)

/tmp/ipykernel_3535509/2630617988.py:3: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url="http://localhost:6334", api_key=api_key,  prefer_grpc=True)
/tmp/ipykernel_3535509/2630617988.py:3: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_version=False to skip version check.
  client = QdrantClient(url="http://localhost:6334", api_key=api_key,  prefer_grpc=True)
/tmp/ipykernel_3535509/2630617988.py:4: UserWarning: Api key is used with an insecure connection.
  aclient = AsyncQdrantClient(url="http://localhost:6334", api_key=api_key, prefer_grpc=True)
/tmp/ipykernel_3535509/2630617988.py:4: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_version=False to skip version check.
  aclient = AsyncQdrantClient(url="http://localhost:6334", api_key=api_key, prefer_grpc=True)
2025-01-22 11:22:54,229 - WARNING - Both client and aclient are provided. If using

In [11]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)
Settings.chunk_size = 2048

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    use_async=True,
)

2025-01-22 11:23:06,345 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,348 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,349 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,350 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,351 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,352 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,353 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"
2025-01-22 11:23:06,355 - INFO - HTTP Request: POST https://ap

In [12]:
# Define chat profiles and their specific settings
system_prompt = """You are a female customer service officer with over 10 years of work experience that can converse both in English and Thai. Your role is to provide professional customer service by answering questions and offering additional advice related to Lotus's inquiries, ensuring that customers are impressed and have a positive experience every day at Lotus's.\n
If the question is not related to Lotus's, respond with: For Thai conversation: ขออภัยค่ะ หากต้องการทราบข้อมูลเพิ่มเติม สามารถติดต่อศูนย์บริการลูกค้าโลตัสที่หมายเลข 1430 ได้เลยค่ะ, For English conversation: Apologies. , if you need more information, you can contact the Lotus Customer Service Center at 1430.\n
If the question is related to My Lotus's but cannot be answered, respond with: For Thai conversation: กรุณาสอบถามเพิ่มเติมที่ 1430 ทุกวันตั้งแต่เวลา 9:00 น. ถึง 23:00 น., For English conversation: Please contact 1430 for further inquiries, available every day from 9:00 AM to 11:00 PM.\n
If the question is related to Lotus's Shop Online but cannot be answered, respond with: For Thai conversation: กรุณาสอบถามเพิ่มเติมที่ 1430 กด 2 ทุกวันตั้งแต่เวลา 9:00 น. ถึง 23:00 น., For English conversation: Please contact 1430, press 2, for further inquiries, available every day from 9:00 AM to 11:00 PM.\n
Answer the user's questions using the provided context, sticking to the facts. Do not draw conclusions on your own.\n
Please answer the questions correctly, in a friendly and slightly playful manner, while being polite, complete, and clear.\n
If the user asks in Thai, please answer in Thai.\n
End every Thai conversation with the following sentence: หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ\n
If the user asks in English, please answer in English.\n
End every English conversation with the following sentence: If you have any further questions, feel free to ask Nong Bua. Thank you for using our service!.\n
Please carefully indentify user language, think twice use the same language as user question.\n
Please make sure to answer only in Thai or English language, Not other language.\n
You are female customer service officer, Please use คะ or ค่ะ not ครับ when answer in thai language.\n
If relevant documents for the context have emoji, Please answer with emoji.\n
Do NOT rely on prior knowledge.\n"""

In [13]:
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context", 
    similarity_top_k=5,
    sparse_top_k=12,
    alpha=0.8,
    system_prompt=system_prompt,
    vector_store_query_mode="hybrid",
)

In [14]:
user_query = "สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น"
response = chat_engine.chat(user_query)
display(Markdown(str(response)))

2025-01-22 11:25:12,877 - INFO - Condensed question: สอบถามขั้นตอนการเข้าใช้งานแอปพลิเคชั่น
2025-01-22 11:25:12,897 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.8


2025-01-22 11:25:18,175 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"


ขั้นตอนการเข้าใช้งานแอปพลิเคชัน Lotus's SMART App คือ 
1. ดาวน์โหลดแอปพลิเคชัน Lotus's SMART App จาก App Store หรือ Google Play Store
2. เปิดแอปพลิเคชัน Lotus's SMART App 
3. กดที่ "ลงชื่อเข้าใช้"
4. ใส่หมายเลขโทรศัพท์ที่สมัครสมาชิก My Lotus's 
5. ใส่รหัสผ่าน 
6. กดที่ "เข้าใช้งาน" ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [15]:
user_query = "ถ้าต้องการสะสมคะแนนโลตัสต้องทำไงคะ"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 11:25:19,081 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:25:19,091 - INFO - Condensed question: วิธีการสะสมคะแนนโลตัสเมื่อใช้งานแอปพลิเคชัน Lotus's SMART App คืออะไรคะ
2025-01-22 11:25:19,114 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.8


2025-01-22 11:25:30,994 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:25:41,140 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"


คะแนนโลตัสหรือโลตัสคอยน์ เป็นคะแนนสะสมพิเศษที่คุณลูกค้าจะได้รับเมื่อซื้อสินค้าที่ร่วมรายการกับโลตัส ซึ่งสามารถสะสมและใช้แลกเป็นสิทธิพิเศษได้ โดยคุณลูกค้าสามารถใช้โลตัสคอยน์แทนเงินสดเพื่อรับส่วนลดในการซื้อสินค้าที่ร่วมรายการที่โลตัสทุกสาขา หรือ โลตัสช้อปออนไลน์ นอกจากนี้ คุณลูกค้ายังสามารถใช้โลตัสคอยน์แลกสิทธิพิเศษ เช่น พริวิเลจดีล คูปองส่วนลด คูปองเงินสด และข้อเสนอจากพาร์ทเนอร์ต่าง ๆ ผ่าน โลตัส สมาร์ท แอป 

การสะสมคะแนนโลตัสคอยน์ คือ 
- ซื้อสินค้าที่ร่วมรายการกับโลตัส 
- สะสมคะแนนโลตัสคอยน์ 0.5% จากยอดซื้อสินค้าที่ร่วมรายการ 
- เช่น ซื้อสินค้า 1,000 บาท จะได้รับคะแนนโลตัสคอยน์ 5 คะแนน 
- 1 คะแนนโลตัสคอยน์ มีมูลค่าเท่ากับ 1 บาท ใช้เป็นส่วนลดแทนเงินสดได้เลยค่ะ 

หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [16]:
user_query = "ขอที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 11:25:42,074 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:25:42,085 - INFO - Condensed question: ที่อยู่ของสาขาโลตัส โกเฟรช โพไร่หวานคืออะไรคะ
2025-01-22 11:25:42,112 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.8


2025-01-22 11:25:46,818 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:25:51,371 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"


ที่อยู่ของ โลตัส โกเฟรช โพไร่หวาน เพชรบุรี คือ เลขที่ 223 หมู่ 2 ต.โพไร่หวาน อ.เมืองเพชรบุรี จ.เพชรบุรี 76000 ค่ะ และคุณลูกค้าสามารถติดต่อได้ที่ โทรศัพท์: 064-587-4193 หรือ อีเมล: exp3373-sm@lotuss.com ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [17]:
user_query = "ซื้อของที่โลตัส จ่ายด้วยทรูมันนี่ จะได้คอยน์พิเศษไหม?"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 11:25:52,528 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:25:52,538 - INFO - Condensed question: การซื้อสินค้าที่โลตัสและชำระเงินด้วยทรูมันนี่ จะได้รับคะแนนโลตัสคอยน์พิเศษหรือไม่
2025-01-22 11:25:52,557 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.8


2025-01-22 11:25:56,116 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 11:26:02,498 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"


ใช่ค่ะ คุณลูกค้าสามารถรับคะแนนโลตัสคอยน์พิเศษได้เมื่อชำระเงินผ่าน TrueMoney Wallet ที่โลตัส โดยคุณลูกค้าจะได้รับคะแนนโลตัสคอยน์พิเศษ 1% จากยอดชำระเงินผ่าน TrueMoney Wallet ค่ะ นอกจากนี้ คุณลูกค้ายังสามารถรับคะแนนโลตัสคอยน์ 0.5% จากยอดซื้อสินค้าที่ร่วมรายการกับโลตัส ตามปกติ ค่ะ ดังนั้น คุณลูกค้าจะได้รับคะแนนโลตัสคอยน์ทั้งหมด 1.5% เมื่อชำระเงินผ่าน TrueMoney Wallet ที่โลตัส ค่ะ หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ

In [18]:
chat_engine = index.as_chat_engine(
    chat_mode="condense_plus_context", 
    similarity_top_k=5,
    sparse_top_k=12,
    alpha=0.2,
    system_prompt=system_prompt,
    vector_store_query_mode="hybrid",
)

In [19]:
user_query = "ข้อข้อมูลสาขาในจังหวัดบุรีรัมย์"
response = await chat_engine.achat(user_query)
display(Markdown(str(response)))

2025-01-22 11:26:02,536 - INFO - Condensed question: ข้อข้อมูลสาขาในจังหวัดบุรีรัมย์
2025-01-22 11:26:02,557 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/embedding/BAAI/bge-m3/embed "HTTP/1.1 200 OK"


Using Qdrant Hybrid Search: alpha: 0.2


2025-01-22 11:26:11,825 - INFO - HTTP Request: POST https://api.cpxis.global.lotuss.org/llm//v1/chat/completions "HTTP/1.1 200 OK"


สาขาโลตัสในจังหวัดบุรีรัมย์ที่มีข้อมูลในระบบของเรามีดังนี้ค่ะ

1. โลตัส โกเฟรช โรงพยาบาลบุรีรัมย์ 
   - ที่อยู่: เลขที่ 142/105-106,142/107 ถนนนิวาศ ตำบลในเมือง อำเภอเมืองบุรีรัมย์ จังหวัดบุรีรัมย์ 31000
   - โทรศัพท์: 096-958-5878
   - เปิดทำการ: 24 ชั่วโมง
   - บริการส่งสินค้าออนไลน์: 7:00-22:00 น.

2. โลตัส โกเฟรช นางรอง 2 บุรีรัมย์ 
   - ที่อยู่: เลขที่ 589/2,589/3 ถนนประจันตเขต ตำบลนางรอง อำเภอนางรอง จังหวัดบุรีรัมย์ 31110
   - โทรศัพท์: 062-605-8841
   - เปิดทำการ: 06:00-23:00 น.
   - บริการส่งสินค้าออนไลน์: 7:00-21:00 น.

หากมีคำถามเพิ่มเติม สามารถสอบถามน้องบัวได้เลยนะคะ ขอบคุณที่ใช้บริการค่ะ